# COMP5329 - Deep Learning
**Tutorial 4 — Regularization for Deep Models**  
**Semester 1, 2026**

**Objectives:**
- Understand overfitting and how it manifests on a non-trivial dataset
- Extend the Week 3 MLP framework with regularization options as constructor flags
- Implement and compare four regularization strategies from scratch:
  - **Weight Decay** (L2 penalty inside the optimiser)
  - **Dropout** (inverted dropout with train/eval mode)
  - **Batch Normalisation** (with learnable γ, β and full backward pass)
  - **Early Stopping** (validation-loss patience with weight snapshot/restore)

---
## 0. From Week 3 — Base Framework

The following cells reproduce the Week 3 components unchanged: `Dataset`, `DataLoader`, `init_weights`, `Activation`, `softmax`, and `CrossEntropyLoss`. We build on top of them in §3.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import math, copy

%matplotlib inline
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── Dataset & DataLoader (Week 3) ──────────────────────────────────
class Dataset:
    def __len__(self): raise NotImplementedError
    def __getitem__(self, idx): raise NotImplementedError

class DataLoader:
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset; self.batch_size = batch_size; self.shuffle = shuffle
    def __len__(self):
        return math.ceil(len(self.dataset) / self.batch_size)
    def __iter__(self):
        n = len(self.dataset)
        indices = torch.randperm(n) if self.shuffle else torch.arange(n)
        for start in range(0, n, self.batch_size):
            idx = indices[start : start + self.batch_size]
            samples = [self.dataset[i.item()] for i in idx]
            yield torch.stack([s[0] for s in samples]), torch.stack([s[1] for s in samples])

class PointDataset(Dataset):
    def __init__(self, X, y): self.X = X; self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

In [ ]:
# ── Weight initialisation (Week 3) ─────────────────────────────────
def init_weights(method, shape):
    fan_in, fan_out = shape
    if method == "zeros":   return torch.zeros(*shape)
    elif method == "xavier": return torch.randn(*shape) * math.sqrt(2.0/(fan_in+fan_out))
    elif method == "he":     return torch.randn(*shape) * math.sqrt(2.0/fan_in)
    else: raise ValueError(f"Unknown init: {method}")

# ── Activation (Week 3) ─────────────────────────────────────────────
class Activation:
    def __init__(self, name="relu"): self.name = name; self._z = None
    def forward(self, z):
        self._z = z
        if self.name == "sigmoid": return 1.0/(1.0+torch.exp(-z))
        elif self.name == "tanh":  return torch.tanh(z)
        elif self.name == "relu":  return z.clamp(min=0)
        else: return z
    def backward(self, grad):
        z = self._z
        if self.name == "sigmoid": s=1.0/(1.0+torch.exp(-z)); return grad*s*(1.0-s)
        elif self.name == "tanh":  return grad*(1.0-torch.tanh(z)**2)
        elif self.name == "relu":  return grad*(z>0).float()
        else: return grad

# ── Loss (Week 3) ────────────────────────────────────────────────────
def softmax(logits):
    e = torch.exp(logits - logits.max(dim=1,keepdim=True).values)
    return e / e.sum(dim=1,keepdim=True)

class CrossEntropyLoss:
    def __init__(self): self._probs = self._labels = None
    def forward(self, logits, labels):
        self._probs = softmax(logits); self._labels = labels
        N = labels.shape[0]
        return -torch.log(self._probs[torch.arange(N), labels] + 1e-9).mean()
    def backward(self):
        N = self._labels.shape[0]
        g = self._probs.clone()
        g[torch.arange(N), self._labels] -= 1.0
        return g / N

---
## 1. A Dataset Designed to Expose Overfitting

The Gaussian blobs from Week 3 are too easy — a small model solves them with little risk of overfitting. Here we use a **two-class 2D spiral** dataset:

- Two interleaving spiral arms, each with Gaussian noise
- **Small training set** (60 samples) relative to the model capacity → easy to overfit
- **Larger validation/test set** (300 samples) to measure generalisation

The non-linear boundary means a shallow or linear model cannot fit the data, while a large MLP will memorise the training set if unconstrained.

In [ ]:
def make_spiral(n_per_class=150, noise=0.25, seed=0):
    """Generate a 2-class 2D spiral dataset.

    Args:
        n_per_class : points per spiral arm
        noise       : Gaussian noise std added to coordinates
    Returns:
        X : (2*n_per_class, 2) float32 tensor
        y : (2*n_per_class,)  long   tensor  (0 or 1)
    """
    torch.manual_seed(seed)
    N     = n_per_class
    theta = torch.linspace(0, 4 * math.pi, N)
    r     = torch.linspace(0.4, 2.2, N)

    X0 = torch.stack([r * torch.cos(theta),
                       r * torch.sin(theta)], dim=1)
    X1 = torch.stack([r * torch.cos(theta + math.pi),
                       r * torch.sin(theta + math.pi)], dim=1)
    X0 += torch.randn_like(X0) * noise
    X1 += torch.randn_like(X1) * noise

    X = torch.cat([X0, X1])
    y = torch.cat([torch.zeros(N, dtype=torch.long),
                   torch.ones( N, dtype=torch.long)])
    perm = torch.randperm(2 * N)
    return X[perm], y[perm]


# ── Generate full dataset then split ────────────────────────────────
X_all, y_all = make_spiral(n_per_class=180, noise=0.25)

N_TRAIN = 60          # deliberately small → easy to overfit
N_VAL   = 120
# rest goes to test

X_train, y_train = X_all[:N_TRAIN],        y_all[:N_TRAIN]
X_val,   y_val   = X_all[N_TRAIN:N_TRAIN+N_VAL], y_all[N_TRAIN:N_TRAIN+N_VAL]
X_test,  y_test  = X_all[N_TRAIN+N_VAL:], y_all[N_TRAIN+N_VAL:]

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles  = ["Train set", "Validation set", "Test set"]
splits  = [(X_train, y_train), (X_val, y_val), (X_test, y_test)]

for ax, (Xs, ys), title in zip(axes, splits, titles):
    ax.scatter(Xs[ys==0, 0], Xs[ys==0, 1], c="steelblue",
               edgecolors="k", linewidths=0.3, s=28, label="Class 0")
    ax.scatter(Xs[ys==1, 0], Xs[ys==1, 1], c="tomato",
               edgecolors="k", linewidths=0.3, s=28, label="Class 1")
    ax.set_title(f"{title}  (n={len(Xs)})")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2); ax.axis("equal")

plt.suptitle("Spiral Dataset — Train / Val / Test Splits", fontsize=13)
plt.tight_layout(); plt.show()

---
## 2. Overfitting Illustration

**Overfitting** occurs when a model learns the training data too well — including its noise — at the expense of generalisation to unseen data. It is diagnosed by:

- Training loss keeps decreasing
- Validation loss starts increasing (or stops improving) after a certain point
- Decision boundary becomes jagged / over-complex

We reproduce the **plain Week 3 framework** here (no regularization) and deliberately use a network that is far too large for 60 training points.

In [ ]:
# ── Plain HiddenLayer from Week 3 (no regularization) ──────────────
class HiddenLayer:
    def __init__(self, n_in, n_out, activation="relu", initialization="xavier"):
        self.W   = init_weights(initialization, (n_in, n_out))
        self.b   = torch.zeros(1, n_out)
        self.act = Activation(activation)
        self._x  = None
    def forward(self, x):
        self._x = x
        return self.act.forward(x @ self.W + self.b)
    def backward(self, grad):
        dz = self.act.backward(grad)
        dW = self._x.T @ dz; db = dz.sum(0,keepdim=True); dx = dz @ self.W.T
        return dx, dW, db

class MLP:
    def __init__(self, layer_dims, activations, initialization="xavier"):
        self.layers = [HiddenLayer(layer_dims[i], layer_dims[i+1],
                                   activations[i], initialization)
                        for i in range(len(layer_dims)-1)]
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def backward(self, grad):
        pg = []
        for l in reversed(self.layers): grad,dW,db=l.backward(grad); pg.append((dW,db))
        return list(reversed(pg))

class SGD:
    def __init__(self, lr=0.01): self.lr = lr
    def step(self, layers, grads):
        for l,(dW,db) in zip(layers,grads): l.W-=self.lr*dW; l.b-=self.lr*db

def train_plain(model, optimizer, X_tr, y_tr, X_val, y_val,
                n_epochs=1000, batch_size=16):
    """Simple train loop — records both train and val loss."""
    tr_loader = DataLoader(PointDataset(X_tr, y_tr), batch_size, shuffle=True)
    crit      = CrossEntropyLoss()
    tr_losses, val_losses = [], []
    for epoch in range(1, n_epochs+1):
        el = 0.0
        for Xb, yb in tr_loader:
            loss  = crit.forward(model.forward(Xb), yb); el += loss.item()
            grads = model.backward(crit.backward())
            optimizer.step(model.layers, grads)
        tr_losses.append(el / len(tr_loader))
        with torch.no_grad():
            val_losses.append(crit.forward(model.forward(X_val), y_val).item())
    return tr_losses, val_losses

In [ ]:
torch.manual_seed(0)
# A very large model — 4 hidden layers × 128 neurons on only 60 training points
overfit_model = MLP([2, 128, 128, 128, 128, 2],
                    ["relu"]*4 + ["linear"], initialization="he")
overfit_opt   = SGD(lr=0.05)

print("Training overfit model (no regularisation)...")
tr_loss, val_loss = train_plain(overfit_model, overfit_opt,
                                X_train, y_train, X_val, y_val,
                                n_epochs=1000, batch_size=16)

print(f"Final  train loss : {tr_loss[-1]:.4f}")
print(f"Final  val   loss : {val_loss[-1]:.4f}")

In [ ]:
def plot_boundary(model, X, y, title="", ax=None, training=True):
    """Helper: render probability contour over 2D feature space."""
    x_min,x_max = X[:,0].min()-0.3, X[:,0].max()+0.3
    y_min,y_max = X[:,1].min()-0.3, X[:,1].max()+0.3
    xx,yy = np.meshgrid(np.arange(x_min,x_max,0.04),
                         np.arange(y_min,y_max,0.04))
    grid  = torch.tensor(np.c_[xx.ravel(),yy.ravel()], dtype=torch.float32)
    # support both plain MLP and extended MLP with set_training
    if hasattr(model, "set_training"): model.set_training(False)
    Z = softmax(model.forward(grid))[:,1].detach().numpy().reshape(xx.shape)
    if hasattr(model, "set_training"): model.set_training(training)
    if ax is None: fig,ax = plt.subplots(figsize=(5,4))
    cf = ax.contourf(xx,yy,Z, levels=50, cmap="RdBu", alpha=0.8)
    plt.colorbar(cf,ax=ax,label="P(class 1)")
    ax.scatter(X[y==0,0],X[y==0,1],c="steelblue",edgecolors="k",lw=0.4,s=20,label="Class 0")
    ax.scatter(X[y==1,0],X[y==1,1],c="tomato",   edgecolors="k",lw=0.4,s=20,label="Class 1")
    ax.set_title(title,fontsize=10); ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# Loss curves
axes[0].plot(tr_loss,  label="Train", color="steelblue", lw=2)
axes[0].plot(val_loss, label="Val",   color="tomato",    lw=2, linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Overfitting: train ↓  but  val ↑"); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Decision boundaries
plot_boundary(overfit_model, X_train, y_train,
              title="Decision boundary on TRAIN set", ax=axes[1])
plot_boundary(overfit_model, X_val,   y_val,
              title="Decision boundary on VAL set",   ax=axes[2])

plt.suptitle("No Regularisation — Clear Overfitting", fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

---
## 3. Unified Regularised MLP Framework

This section defines the **complete extended framework** once. Every technique is an optional constructor/call flag — passing `None` / `False` gives the plain Week 3 behaviour.

> **How to read §3:** You do not need to absorb every line right now. Sections §4–7 each open with a **"Key Implementation" block** that zooms in on the exact lines responsible for that technique, with theory alongside. Come back here for the complete picture.

```python
MLP([2,128,128,128,128,2], [...], dropout_rate=0.5)  # Dropout
MLP([2,128,128,128,128,2], [...], batch_norm=True)   # Batch Norm
Adam(lr=1e-3, weight_decay=1e-3)                    # Weight Decay
train(..., X_val=X_val, y_val=y_val, patience=30)   # Early Stopping
```

### 3.1 Extended `HiddenLayer`

**Dropout (inverted):** During training, each neuron is independently zeroed out with probability $p$. Surviving activations are scaled by $\frac{1}{1-p}$ so that the expected magnitude is unchanged — this is *inverted dropout*, the standard in PyTorch. During evaluation, dropout is simply skipped (no scaling needed).

**Batch Normalisation:** Normalises each feature across the batch before the activation function, then rescales with learnable parameters $\gamma$ (scale) and $\beta$ (shift):

$$\hat{z} = \frac{z - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \qquad \text{BN}(z) = \gamma \odot \hat{z} + \beta$$

The backward pass for BN follows the compact formula:

$$\frac{\partial L}{\partial z_i} = \frac{1}{B \cdot \sigma_B}\left(B\,\delta_i - \sum_j \delta_j - \hat{z}_i \sum_j \delta_j \hat{z}_j\right), \quad \delta_i = \frac{\partial L}{\partial \hat{z}_i} = \frac{\partial L}{\partial y_i} \cdot \gamma$$

In [ ]:
class HiddenLayer:
    """Fully-connected layer with optional Dropout and Batch Normalisation.

    Args:
        n_in           : input features
        n_out          : output neurons
        activation     : activation name   (default "relu")
        initialization : weight init name  (default "xavier")
        dropout_rate   : float in (0,1) or None — dropout probability
        batch_norm     : bool — whether to apply BN before activation
    """

    _BN_EPS = 1e-5

    def __init__(self, n_in, n_out,
                 activation="relu", initialization="xavier",
                 dropout_rate=None, batch_norm=False):
        # Core parameters
        self.W   = init_weights(initialization, (n_in, n_out))
        self.b   = torch.zeros(1, n_out)
        self.act = Activation(activation)
        # Regularisation flags
        self.dropout_rate = dropout_rate
        self.batch_norm   = batch_norm
        # BN learnable parameters (only allocated if needed)
        if batch_norm:
            self.gamma    = torch.ones( 1, n_out)  # scale
            self.beta     = torch.zeros(1, n_out)  # shift
            self._bn_cache = {}
        # Forward-pass caches
        self._x        = None
        self._do_mask  = None   # dropout mask
        self._training = True   # set by MLP.set_training()
        # Accumulated BN gradients (read by optimiser)
        self._dgamma   = None
        self._dbeta    = None

    # ── Forward ──────────────────────────────────────────────────────
    def forward(self, x):
        self._x = x
        z = x @ self.W + self.b               # (B, n_out)  linear

        # ── Batch Normalisation (before activation) ──────────────────
        if self.batch_norm:
            z = self._bn_forward(z)

        # ── Activation ───────────────────────────────────────────────
        out = self.act.forward(z)              # (B, n_out)

        # ── Dropout (after activation, training only) ────────────────
        if self.dropout_rate is not None and self._training:
            # Inverted dropout: keep mask is 1 with prob (1-p), then scale
            keep = (torch.rand_like(out) > self.dropout_rate).float()
            self._do_mask = keep / (1.0 - self.dropout_rate)
            out = out * self._do_mask
        else:
            self._do_mask = None

        return out

    # ── Backward ─────────────────────────────────────────────────────
    def backward(self, grad):
        # ── Dropout backward ─────────────────────────────────────────
        if self._do_mask is not None:
            grad = grad * self._do_mask

        # ── Activation backward ──────────────────────────────────────
        dz = self.act.backward(grad)           # (B, n_out)

        # ── BN backward ───────────────────────────────────────────────
        if self.batch_norm:
            dz = self._bn_backward(dz)

        # ── Linear backward ───────────────────────────────────────────
        x  = self._x
        dW = x.T  @ dz
        db = dz.sum(dim=0, keepdim=True)
        dx = dz   @ self.W.T
        return dx, dW, db

    # ── Batch Norm helpers ────────────────────────────────────────────
    def _bn_forward(self, z):
        mu    = z.mean(dim=0, keepdim=True)                   # (1, n_out)
        var   = z.var( dim=0, keepdim=True, unbiased=False)   # (1, n_out)
        std   = torch.sqrt(var + self._BN_EPS)                # (1, n_out)
        z_hat = (z - mu) / std                                # (B, n_out)
        self._bn_cache = {"z_hat": z_hat, "std": std}
        return self.gamma * z_hat + self.beta

    def _bn_backward(self, grad):
        """Compact BN backward: dz = (1/B*std) * (B*dz_hat - sum(dz_hat)
                                                   - z_hat * sum(dz_hat*z_hat))
        where dz_hat = grad * gamma.
        """
        z_hat = self._bn_cache["z_hat"]        # (B, n_out)
        std   = self._bn_cache["std"]           # (1, n_out)
        B     = z_hat.shape[0]

        # Gradients for learnable BN params (stored for optimiser)
        self._dgamma = (grad * z_hat).sum(dim=0, keepdim=True)
        self._dbeta  =  grad.sum(       dim=0, keepdim=True)

        dz_hat = grad * self.gamma              # (B, n_out)
        dz = (1.0 / (B * std)) * (
            B * dz_hat
            - dz_hat.sum(dim=0, keepdim=True)
            - z_hat * (dz_hat * z_hat).sum(dim=0, keepdim=True)
        )
        return dz

### 3.2 Extended `MLP`

`dropout_rate` and `batch_norm` are forwarded to every **hidden** layer (all layers except the last). The output layer always uses `activation="linear"` with no dropout or BN — these would corrupt the logits.

`set_training(bool)` propagates the training flag to all layers simultaneously, so dropout is automatically disabled during validation and testing.

In [ ]:
class MLP:
    """MLP assembled from the extended HiddenLayer.

    Args:
        layer_dims     : [input_dim, h1, h2, ..., output_dim]
        activations    : one activation name per transition
        initialization : shared weight init strategy
        dropout_rate   : applied to hidden layers only (None = off)
        batch_norm     : applied to hidden layers only (False = off)
    """

    def __init__(self, layer_dims, activations,
                 initialization="xavier",
                 dropout_rate=None, batch_norm=False):
        n_layers = len(layer_dims) - 1
        self.layers = []
        for i in range(n_layers):
            is_output = (i == n_layers - 1)
            self.layers.append(HiddenLayer(
                layer_dims[i], layer_dims[i+1],
                activation    = activations[i],
                initialization= initialization,
                # Regularisation only on hidden layers, not the output layer
                dropout_rate  = None         if is_output else dropout_rate,
                batch_norm    = False         if is_output else batch_norm,
            ))

    def set_training(self, mode: bool):
        """Propagate training / eval mode to all layers (affects dropout)."""
        for layer in self.layers:
            layer._training = mode

    def forward(self, x):
        for layer in self.layers: x = layer.forward(x)
        return x

    def backward(self, grad):
        pg = []
        for layer in reversed(self.layers):
            grad, dW, db = layer.backward(grad)
            pg.append((dW, db))
        return list(reversed(pg))

    def snapshot(self):
        """Deep-copy all parameters — used for early stopping."""
        snap = []
        for l in self.layers:
            entry = {"W": l.W.clone(), "b": l.b.clone()}
            if l.batch_norm:
                entry["gamma"] = l.gamma.clone()
                entry["beta"]  = l.beta.clone()
            snap.append(entry)
        return snap

    def restore(self, snap):
        """Restore parameters from a snapshot."""
        for l, entry in zip(self.layers, snap):
            l.W = entry["W"]; l.b = entry["b"]
            if l.batch_norm:
                l.gamma = entry["gamma"]; l.beta = entry["beta"]

### 3.3 Extended Optimisers — Weight Decay

**Weight Decay (L2 regularisation)** adds a penalty $\frac{\lambda}{2}\|W\|^2$ to the loss. Taking the derivative w.r.t. $W$:

$$\nabla_W \mathcal{L}_{\text{reg}} = \nabla_W \mathcal{L} + \lambda W$$

In the optimiser, this means: before multiplying by the learning rate, add $\lambda W$ to every weight gradient. Biases are **not** penalised (standard practice), and neither are BN parameters $\gamma, \beta$.

> Tip: weight decay and dropout address different failure modes. Weight decay shrinks weights towards zero (discourages large weights). Dropout forces the network to not rely on any single neuron.

In [ ]:
class SGD:
    """SGD with optional weight decay and BN param updates.

    Args:
        lr           : learning rate
        weight_decay : L2 penalty coefficient (default 0 = off)
    """
    def __init__(self, lr=0.01, weight_decay=0.0):
        self.lr = lr; self.wd = weight_decay

    def step(self, layers, grads):
        for layer, (dW, db) in zip(layers, grads):
            # L2 penalty: add lambda*W to the weight gradient
            layer.W -= self.lr * (dW + self.wd * layer.W)
            layer.b -= self.lr * db
            # Update BN learnable params (no weight decay on gamma/beta)
            if layer.batch_norm and layer._dgamma is not None:
                layer.gamma -= self.lr * layer._dgamma
                layer.beta  -= self.lr * layer._dbeta


class Adam:
    """Adam with optional weight decay.

    Args:
        lr           : learning rate  (default 1e-3)
        beta1/beta2  : moment decay   (default 0.9 / 0.999)
        eps          : numerical eps  (default 1e-8)
        weight_decay : L2 penalty     (default 0 = off)
    """
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999,
                 eps=1e-8, weight_decay=0.0):
        self.lr=lr; self.b1=beta1; self.b2=beta2
        self.eps=eps; self.wd=weight_decay; self.t=0
        self._mW=self._mb=self._vW=self._vb=None

    def step(self, layers, grads):
        if self._mW is None:
            self._mW=[torch.zeros_like(l.W) for l in layers]
            self._mb=[torch.zeros_like(l.b) for l in layers]
            self._vW=[torch.zeros_like(l.W) for l in layers]
            self._vb=[torch.zeros_like(l.b) for l in layers]
        self.t += 1
        for i,(layer,(dW,db)) in enumerate(zip(layers,grads)):
            # Apply weight decay before moment update
            dW_reg = dW + self.wd * layer.W
            self._mW[i]=self.b1*self._mW[i]+(1-self.b1)*dW_reg
            self._mb[i]=self.b1*self._mb[i]+(1-self.b1)*db
            self._vW[i]=self.b2*self._vW[i]+(1-self.b2)*dW_reg**2
            self._vb[i]=self.b2*self._vb[i]+(1-self.b2)*db**2
            mWh=self._mW[i]/(1-self.b1**self.t); mbh=self._mb[i]/(1-self.b1**self.t)
            vWh=self._vW[i]/(1-self.b2**self.t); vbh=self._vb[i]/(1-self.b2**self.t)
            layer.W -= self.lr * mWh / (torch.sqrt(vWh)+self.eps)
            layer.b -= self.lr * mbh / (torch.sqrt(vbh)+self.eps)
            # BN params
            if layer.batch_norm and layer._dgamma is not None:
                layer.gamma -= self.lr * layer._dgamma
                layer.beta  -= self.lr * layer._dbeta

### 3.4 Extended `train()` — Validation Tracking and Early Stopping

**Early Stopping** halts training when the validation loss has not improved for `patience` consecutive epochs. It also **restores the best weights** seen during training, so the returned model is the one that performed best on the validation set — not the last checkpoint.

Implementation uses `MLP.snapshot()` / `MLP.restore()` to deep-copy parameters.

In [ ]:
def train(model, criterion, optimizer, X_tr, y_tr,
          X_val=None, y_val=None, patience=None,
          n_epochs=1000, batch_size=16, verbose=True):
    """Training loop with validation tracking and optional early stopping.

    Args:
        model      : MLP instance
        criterion  : CrossEntropyLoss instance
        optimizer  : SGD or Adam instance
        X_tr/y_tr  : training tensors
        X_val/y_val: validation tensors (required for early stopping)
        patience   : epochs without val improvement before stopping (None=off)
        n_epochs   : maximum training epochs
        batch_size : mini-batch size
        verbose    : print every 200 epochs

    Returns:
        tr_losses  : list of per-epoch train loss
        val_losses : list of per-epoch val loss  (empty if no val set)
    """
    loader = DataLoader(PointDataset(X_tr, y_tr), batch_size, shuffle=True)
    tr_losses, val_losses = [], []

    best_val  = float("inf")
    best_snap = None
    no_imp    = 0            # epochs since last improvement

    for epoch in range(1, n_epochs + 1):

        # ── Training step ─────────────────────────────────────────────
        model.set_training(True)
        el = 0.0
        for Xb, yb in loader:
            loss  = criterion.forward(model.forward(Xb), yb)
            el   += loss.item()
            grads = model.backward(criterion.backward())
            optimizer.step(model.layers, grads)
        tr_losses.append(el / len(loader))

        # ── Validation step ───────────────────────────────────────────
        if X_val is not None:
            model.set_training(False)
            vl = criterion.forward(model.forward(X_val), y_val).item()
            val_losses.append(vl)

            # Early stopping logic
            if patience is not None:
                if vl < best_val - 1e-5:
                    best_val  = vl
                    best_snap = model.snapshot()  # save best weights
                    no_imp    = 0
                else:
                    no_imp += 1
                    if no_imp >= patience:
                        if verbose:
                            print(f"Early stop at epoch {epoch} "
                                  f"(best val={best_val:.4f})")
                        model.restore(best_snap)  # restore best weights
                        break

        if verbose and epoch % 200 == 0:
            msg = f"Epoch {epoch:4d}  |  train={tr_losses[-1]:.4f}"
            if val_losses: msg += f"  val={val_losses[-1]:.4f}"
            print(msg)

    return tr_losses, val_losses

In [ ]:
def make_model(dropout_rate=None, batch_norm=False, seed=0):
    """Create a fresh model with reproducible weights."""
    torch.manual_seed(seed)
    return MLP(
        layer_dims   = [2, 128, 128, 128, 128, 2],
        activations  = ["relu", "relu", "relu", "relu", "linear"],
        initialization = "he",
        dropout_rate = dropout_rate,
        batch_norm   = batch_norm,
    )

print("Framework ready. Example model structures:")
for desc, kw in [
    ("Baseline (no reg)",     {}),
    ("Dropout 50%",           {"dropout_rate": 0.5}),
    ("Batch Norm",            {"batch_norm": True}),
]:
    m = make_model(**kw)
    has_do = any(l.dropout_rate is not None for l in m.layers)
    has_bn = any(l.batch_norm              for l in m.layers)
    print(f"  {desc:25s}  dropout={has_do}  bn={has_bn}")

---
## 4. Weight Decay

Weight decay constrains model complexity by keeping weights small. It is equivalent to placing a Gaussian prior over the weights (MAP estimation). The key effect is that it penalises neurons that have grown very large weights, forcing the network to use many small weights rather than a few dominant ones — which reduces sensitivity to individual training examples.

| $\lambda$ | Effect |
|-----------|--------|
| 0 | No regularisation (baseline) |
| 1e-4 | Mild — slight smoothing of boundary |
| 1e-3 | Moderate — clearly smoother boundary |
| 1e-2 | Strong — may underfit |

We pass `weight_decay` to the **optimiser**, not the model constructor.

#### Key Implementation — Weight Decay in the Optimiser

Weight decay lives entirely in `SGD.step()` / `Adam.step()`. Nothing in `HiddenLayer` or `MLP` changes. The only addition is **one extra term per weight tensor** before the update:

```python
# SGD.step()  ← the only change from plain Week-3 SGD
for layer, (dW, db) in zip(layers, grads):
    layer.W -= self.lr * (dW + self.wd * layer.W)  # ← + λW
    layer.b -= self.lr * db                         # bias: no decay

# Adam.step()  ← decay applied before the moment update
    dW_reg = dW + self.wd * layer.W                 # ← + λW
    self._mW[i] = self.b1 * self._mW[i] + (1 - self.b1) * dW_reg
    ...
```

Adding $\lambda W$ to the gradient is equivalent to an L2 penalty $\tfrac{\lambda}{2}\|W\|^2$ in the loss. Biases and BN parameters ($\gamma, \beta$) are **never** penalised — standard practice in all major frameworks.

In [ ]:
model_wd = make_model()
crit_wd  = CrossEntropyLoss()
opt_wd   = Adam(lr=1e-3, weight_decay=1e-3)

print("Training with Weight Decay (λ=1e-3)...")
tr_wd, val_wd = train(model_wd, crit_wd, opt_wd, X_train, y_train,
                      X_val=X_val, y_val=y_val, n_epochs=1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tr_wd,  label="Train",      color="steelblue", lw=2)
axes[0].plot(val_wd, label="Validation", color="tomato",    lw=2, linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Weight Decay — Loss Curves"); axes[0].legend()
axes[0].grid(True, alpha=0.3)
plot_boundary(model_wd, X_val, y_val,
              title="Weight Decay — Val Decision Boundary", ax=axes[1])
plt.tight_layout(); plt.show()

---
## 5. Dropout

Dropout (Srivastava et al., 2014) randomly **silences** each neuron independently during each forward pass with probability $p$. This prevents co-adaptation: neurons cannot rely on specific other neurons being present, so each must learn useful features on its own. The effect is similar to training an ensemble of $2^n$ thinned networks and averaging them at test time.

**Inverted dropout** scales surviving activations by $\frac{1}{1-p}$ during training so that the expected sum is unchanged. During evaluation, no scaling or masking is needed.

| dropout rate $p$ | Effect |
|-----------------|--------|
| 0 | No dropout |
| 0.3 | Mild — 30% neurons zeroed each step |
| 0.5 | Standard choice for fully-connected layers |
| 0.8 | Heavy — may require more epochs to converge |

#### Key Implementation — Dropout in `HiddenLayer`

Dropout adds **7 lines** spread across `__init__`, `forward`, and `backward`. Everything else in `HiddenLayer` is unchanged.

```python
# __init__
self.dropout_rate = dropout_rate   # float ∈ (0,1) or None
self._do_mask     = None           # cache mask for backward

# forward()  — applied AFTER activation
if self.dropout_rate is not None and self._training:
    keep          = (torch.rand_like(out) > self.dropout_rate).float()
    self._do_mask = keep / (1.0 - self.dropout_rate)  # ← inverted scaling
    out           = out * self._do_mask
else:
    self._do_mask = None   # eval mode: skip entirely

# backward()  — before activation.backward()
if self._do_mask is not None:
    grad = grad * self._do_mask   # same mask, same scale
```

The **inverted scaling** `÷ (1-p)` means surviving activations have the same expected magnitude as without dropout, so **no rescaling is needed at eval time**. `MLP.set_training(False)` propagates `_training = False` to all layers.

In [ ]:
model_do = make_model(dropout_rate=0.5)
crit_do  = CrossEntropyLoss()
opt_do   = Adam(lr=1e-3)

print("Training with Dropout (p=0.5)...")
tr_do, val_do = train(model_do, crit_do, opt_do, X_train, y_train,
                      X_val=X_val, y_val=y_val, n_epochs=1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tr_do,  label="Train",      color="steelblue", lw=2)
axes[0].plot(val_do, label="Validation", color="tomato",    lw=2, linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Dropout p=0.5 — Loss Curves"); axes[0].legend()
axes[0].grid(True, alpha=0.3)
plot_boundary(model_do, X_val, y_val,
              title="Dropout — Val Decision Boundary", ax=axes[1])
plt.tight_layout(); plt.show()

---
## 6. Batch Normalisation

Batch Normalisation (Ioffe & Szegedy, 2015) normalises the input to each layer across the mini-batch to have zero mean and unit variance. This reduces **internal covariate shift** — the distribution of each layer's inputs no longer changes during training — which allows higher learning rates and more stable training.

BN also acts as a mild regulariser: because normalisation depends on the mini-batch statistics, there is noise in the normalised activations that acts similarly to dropout. However, its primary effect is **training stability and speed**, not regularisation.

**Learnable parameters** $\gamma$ (scale) and $\beta$ (shift) allow the network to undo the normalisation if it is beneficial. They are initialised to $\gamma=1$, $\beta=0$ so BN starts as pure normalisation.

> **Note:** BN behaves differently during training (uses batch stats) and evaluation (uses running mean/variance). Our implementation always uses batch stats — a simplification sufficient for small batches. Production implementations maintain a running mean/variance for inference.

#### Key Implementation — Batch Normalisation in `HiddenLayer`

BN is the most mathematically involved technique. It sits **between the linear transform and the activation** and splits into two helpers.

**Forward** — normalise over the batch, then rescale with learnable $\gamma, \beta$:

```python
# _bn_forward(z)  — called as:  z → BN(z) → activation
mu    = z.mean(dim=0, keepdim=True)                    # (1, n_out)
var   = z.var( dim=0, keepdim=True, unbiased=False)    # (1, n_out)
std   = torch.sqrt(var + 1e-5)
z_hat = (z - mu) / std                                  # normalised
return self.gamma * z_hat + self.beta                   # scale + shift
# cache: {z_hat, std}  ← needed for backward
```

**Backward** — compact closed-form (chain rule through mean + variance):

```python
# _bn_backward(grad)  — grad = dL/d(BN output)
self._dgamma = (grad * z_hat).sum(dim=0, keepdim=True)  # dL/dγ
self._dbeta  =  grad.sum(       dim=0, keepdim=True)    # dL/dβ
dz_hat = grad * self.gamma
dz = (1.0 / (B * std)) * (
    B * dz_hat
    - dz_hat.sum(dim=0, keepdim=True)                    # remove gradient mean
    - z_hat * (dz_hat * z_hat).sum(dim=0, keepdim=True)  # remove projection
)
```

The three terms in `dz` backpropagate through the mean subtraction, variance scaling, and normalisation simultaneously. The optimiser reads `layer._dgamma` / `layer._dbeta` and updates $\gamma, \beta$ alongside $W$ and $b$.

In [ ]:
model_bn = make_model(batch_norm=True)
crit_bn  = CrossEntropyLoss()
opt_bn   = Adam(lr=1e-3)

print("Training with Batch Normalisation...")
tr_bn, val_bn = train(model_bn, crit_bn, opt_bn, X_train, y_train,
                      X_val=X_val, y_val=y_val, n_epochs=1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tr_bn,  label="Train",      color="steelblue", lw=2)
axes[0].plot(val_bn, label="Validation", color="tomato",    lw=2, linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Batch Norm — Loss Curves"); axes[0].legend()
axes[0].grid(True, alpha=0.3)
plot_boundary(model_bn, X_val, y_val,
              title="Batch Norm — Val Decision Boundary", ax=axes[1])
plt.tight_layout(); plt.show()

---
## 7. Early Stopping

Early stopping is the simplest regularisation: stop training when the validation loss stops improving. It treats the number of training epochs as a hyperparameter and automatically selects a good value.

Our implementation monitors validation loss with a **patience** counter:

```
if val_loss improved:   → save snapshot of best weights,  reset patience counter
else:                   → increment patience counter
                          if counter >= patience: restore best weights and stop
```

| `patience` | Behaviour |
|------------|----------|
| 5 | Stop very quickly after val loss plateaus |
| 20 | Moderate — waits for a stable plateau |
| 50 | Patient — useful when training loss is noisy |

Note: early stopping is **complementary** to other regularisation techniques — it can be combined with dropout, weight decay, or BN simultaneously.

#### Key Implementation — Early Stopping in `train()`

Early stopping adds a **patience counter and weight snapshot** inside the epoch loop. The forward/backward/update logic is unchanged.

```python
# Before the epoch loop
best_val  = float("inf")
best_snap = None
no_imp    = 0           # counter: epochs since last improvement

# Inside each epoch, after computing val loss vl:
if vl < best_val - 1e-5:         # improvement found
    best_val  = vl
    best_snap = model.snapshot() # deep-copy {W, b, γ, β} for all layers
    no_imp    = 0
else:                            # no improvement
    no_imp += 1
    if no_imp >= patience:
        model.restore(best_snap) # rewind to best checkpoint
        break
```

`model.snapshot()` / `model.restore()` do a deep copy of all parameters (including $\gamma, \beta$ if batch norm is on). The returned model always holds the **best-generalising weights**, not the last-epoch weights.

In [ ]:
model_es = make_model()
crit_es  = CrossEntropyLoss()
opt_es   = Adam(lr=1e-3)

print("Training with Early Stopping (patience=30)...")
tr_es, val_es = train(model_es, crit_es, opt_es, X_train, y_train,
                      X_val=X_val, y_val=y_val,
                      patience=30, n_epochs=2000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tr_es,  label="Train",      color="steelblue", lw=2)
axes[0].plot(val_es, label="Validation", color="tomato",    lw=2, linestyle="--")
# Mark the epoch where early stop triggered
stop_ep = len(tr_es)
axes[0].axvline(stop_ep, color="grey", linestyle=":", lw=1.5, label=f"stopped @ {stop_ep}")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CE Loss")
axes[0].set_title("Early Stopping — Loss Curves"); axes[0].legend()
axes[0].grid(True, alpha=0.3)
plot_boundary(model_es, X_val, y_val,
              title="Early Stopping — Val Decision Boundary", ax=axes[1])
plt.tight_layout(); plt.show()

---
## 8. Full Comparison

We now train all five variants from the same random initialisation and compare their validation loss curves and final decision boundaries side by side.

In [ ]:
CONFIGS = [
    # (label,           model kwargs,              optimizer kwargs,  color)
    ("No reg",          {},                        {},                "grey"),
    ("Weight Decay",    {},                        {"weight_decay":1e-3}, "steelblue"),
    ("Dropout 0.5",     {"dropout_rate": 0.5},     {},                "darkorange"),
    ("Batch Norm",      {"batch_norm": True},      {},                "seagreen"),
    ("Early Stop",      {},                        {},                "purple"),
]

results = {}   # label → (model, tr_losses, val_losses)

for label, model_kw, opt_kw, _ in CONFIGS:
    m    = make_model(**model_kw)
    crit = CrossEntropyLoss()
    opt  = Adam(lr=1e-3, **opt_kw)
    patience = 30 if label == "Early Stop" else None
    print(f"── {label} ──")
    tr, vl = train(m, crit, opt, X_train, y_train,
                   X_val=X_val, y_val=y_val,
                   patience=patience, n_epochs=1000, verbose=False)
    results[label] = (m, tr, vl)
    print(f"   final train={tr[-1]:.4f}  val={vl[-1]:.4f}  epochs={len(tr)}")

In [ ]:
# ── Validation loss curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for (label, _, _, color), (_, _, vl) in zip(CONFIGS, results.values()):
    ax.plot(vl, label=label, color=color, lw=2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation CE Loss")
ax.set_title("Regularisation Comparison — Validation Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# ── Decision boundaries (best val performer highlighted) ─────────────
best_label = min(results, key=lambda k: min(results[k][2]) if results[k][2] else float("inf"))
best_model = results[best_label][0]
plot_boundary(best_model, X_val, y_val,
              title=f"Best val: {best_label}", ax=axes[1])
plt.tight_layout(); plt.show()

# ── Summary table ─────────────────────────────────────────────────────
header = f"\n{'Technique':<20s} {'Train loss':>12s} {'Val loss':>10s} {'Epochs':>8s}"
print(header)
print("-" * 54)
for label, (_, tr, vl) in results.items():
    print(f"{label:<20s} {tr[-1]:>12.4f} {min(vl):>10.4f} {len(tr):>8d}")